In [62]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import Callback
import os
import random

# Python random seed
random.seed(0)
# NumPy seed
np.random.seed(0)
# TensorFlow seed
tf.random.set_seed(0)

# ==========================================
# 1. LOAD & COMBINE DATA 
# ==========================================
# Replace these with your actual file names
train_val_file = '../ED_Calculation/2003_2023/results/EURUSD_Close_Fixed_with_difference_2023.csv'
test_file      = '../ED_Calculation/2024_current/results/eurusd_daily_2024_2025_with_difference.csv'

# Suppress annoying TensorFlow log dumps during the loop
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')



df_train_val = pd.read_csv(train_val_file, index_col='Date', parse_dates=True)
df_test      = pd.read_csv(test_file, index_col='Date', parse_dates=True)
df = pd.concat([df_train_val, df_test]).sort_index().dropna(subset=['difference'])

vol_window = 30
df['scaled_difference'] = df['difference'] / df['difference'].rolling(window=vol_window).std()
df.dropna(subset=['scaled_difference'], inplace=True)

expanded_window_size = 30 
X, y, indices = [], [], []
scaled_diffs = df['scaled_difference'].values
raw_diffs    = df['difference'].values

for i in range(len(scaled_diffs) - expanded_window_size):
    X.append(scaled_diffs[i : i + expanded_window_size])
    y.append(raw_diffs[i + expanded_window_size])
    indices.append(df.index[i + expanded_window_size])

X, y, indices = np.array(X), np.array(y), pd.DatetimeIndex(indices)

train_mask = (indices.year >= 2003) & (indices.year <= 2019)
val_mask   = (indices.year >= 2020) & (indices.year <= 2023)
test_mask  = (indices.year >= 2024)

y_train_binary = np.where(y[train_mask] >= 0, 1, 0)
y_val_binary   = np.where(y[val_mask] >= 0, 1, 0)
y_test_binary  = np.where(y[test_mask] >= 0, 1, 0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X[train_mask])
X_val_scaled   = scaler.transform(X[val_mask])
X_test_scaled  = scaler.transform(X[test_mask])

# ==========================================
# 2. THE REUSABLE CUSTOM CHECKPOINT
# ==========================================
class DirectionalAccuracyCheckpoint(Callback):
    def __init__(self, X_val, y_val, filepath):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
        self.filepath = filepath
        self.best_val_acc = 0.0

    def on_epoch_end(self, epoch, logs=None):
        prob_preds = self.model.predict(self.X_val, verbose=0).flatten()
        binary_preds = np.where(prob_preds >= 0.5, 1, 0)
        current_val_acc = np.mean(binary_preds == self.y_val)
        if current_val_acc > self.best_val_acc:
            self.best_val_acc = current_val_acc
            self.model.save(self.filepath)

# ==========================================
# 3. RUN THE 100-SEED MONTE CARLO LOOP
# ==========================================
all_seed_accuracies = []
checkpoint_path = 'temp_best_seed_model.keras'

print("Starting 100-Seed Robustness Simulation (Seeds 0 to 99)...")
print("This will take a few minutes as it trains 100 individual networks...")

for seed in range(100):
    # Enforce absolute seed isolation for this specific loop iteration
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)
    
    # Initialize fresh model weights based on current seed
    model = Sequential([
        Input(shape=(expanded_window_size,)),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.002), loss='binary_crossentropy')
    
    # Reset and pass custom callback
    seed_checkpoint = DirectionalAccuracyCheckpoint(X_val_scaled, y_val_binary, checkpoint_path)
    
    model.fit(
        X_train_scaled, y_train_binary,
        epochs=50, batch_size=128,
        callbacks=[seed_checkpoint], verbose=0
    )
    
    # Evaluate the saved peak-validation model on the unseen test set
    if os.path.exists(checkpoint_path):
        best_model = load_model(checkpoint_path)
        test_probs = best_model.predict(X_test_scaled, verbose=0).flatten()
        test_preds = np.where(test_probs >= 0.5, 1, 0)
        seed_accuracy = np.mean(test_preds == y_test_binary)
        all_seed_accuracies.append(seed_accuracy)
        
        # Keep a clean workspace
        os.remove(checkpoint_path)
    
    # Progress check-in every 10 seeds so you know it's working
    if (seed + 1) % 10 == 0:
        print(f" > Completed {seed + 1}/100 seeds. Current running median: {np.median(all_seed_accuracies):.2%}")

# ==========================================
# 4. FINAL STATISTICAL DISTRIBUTION REPORT
# ==========================================
all_seed_accuracies = np.array(all_seed_accuracies)
final_median = np.median(all_seed_accuracies)
final_mean   = np.mean(all_seed_accuracies)
final_max    = np.max(all_seed_accuracies)
final_min    = np.min(all_seed_accuracies)

print("\n" + "="*55)
print("     MONTE CARLO SEED ACCURACY DISTRIBUTION REPORT     ")
print("="*55)
print(f"Total Unique Models Trained: {len(all_seed_accuracies)}")
print(f"Absolute Worst Seed Accuracy: {final_min:.2%}")
print(f"Absolute Best Seed Accuracy:  {final_max:.2%}")
print("-"*55)
print(f"MEAN DIRECTIONAL ACCURACY:    {final_mean:.2%}")
print(f"MEDIAN DIRECTIONAL ACCURACY:  {final_median:.2%}")
print("="*55)

Starting 100-Seed Robustness Simulation (Seeds 0 to 99)...
This will take a few minutes as it trains 100 individual networks...
 > Completed 10/100 seeds. Current running median: 49.81%
 > Completed 20/100 seeds. Current running median: 49.13%
 > Completed 30/100 seeds. Current running median: 48.93%
 > Completed 40/100 seeds. Current running median: 48.84%
 > Completed 50/100 seeds. Current running median: 48.84%
 > Completed 60/100 seeds. Current running median: 48.84%
 > Completed 70/100 seeds. Current running median: 48.84%
 > Completed 80/100 seeds. Current running median: 48.84%
 > Completed 90/100 seeds. Current running median: 48.84%
 > Completed 100/100 seeds. Current running median: 48.84%

     MONTE CARLO SEED ACCURACY DISTRIBUTION REPORT     
Total Unique Models Trained: 100
Absolute Worst Seed Accuracy: 44.77%
Absolute Best Seed Accuracy:  54.26%
-------------------------------------------------------
MEAN DIRECTIONAL ACCURACY:    49.19%
MEDIAN DIRECTIONAL ACCURACY:  48.8